# OpenVSP: resize the main wing

Loads the `.vsp3` model, changes **only the main wing** (area and aspect ratio, taper kept), and compares the geometry and the parasite drag before and after.

The wing has one section (root → tip). OpenVSP's section values are per half-wing, so `Area = S/2` and `Aspect = AR/2`. The file on disk is not changed unless `SAVE_AS` is set.

In [1]:
import time
from pathlib import Path

import pandas as pd
import openvsp as vsp

VSP3_FILE = "models/20261012_Convent_BM_V3.vsp3"
WING_NAME = "Main Wing"
GEOM_SET = 1                  # 1 = "Shown"
EXCLUDE = ["CG_Marker"]       # removed from the set for the drag analysis

# new wing, total values for both halves (None = keep current value)
NEW_AREA = 0.50               # m²
NEW_AR = 7.0                  # -
NEW_TAPER = None              # tip chord / root chord

SAVE_AS = None                # e.g. "models/20261012_Convent_BM_V3_wing_mod.vsp3"

In [2]:
def load_model(path=VSP3_FILE):
    vsp.ClearVSPModel()
    vsp.ReadVSPFile(str(Path(path).resolve()))
    for name in EXCLUDE:
        for gid in vsp.FindGeomsWithName(name):
            vsp.SetSetFlag(gid, GEOM_SET, False)
    vsp.Update()
    return vsp.FindGeomsWithName(WING_NAME)[0]


def wing_summary(wing_id) -> pd.Series:
    g = lambda name, group: vsp.GetParmVal(wing_id, name, group)
    return pd.Series({
        "area [m²]": g("TotalArea", "WingGeom"),
        "aspect ratio": g("TotalAR", "WingGeom"),
        "span [m]": g("TotalSpan", "WingGeom"),
        "mean chord [m]": g("TotalChord", "WingGeom"),
        "MAC [m]": g("MAC", "WingGeom"),
        "root chord [m]": g("Root_Chord", "XSec_1"),
        "tip chord [m]": g("Tip_Chord", "XSec_1"),
        "taper": g("Taper", "XSec_1"),
    })


def set_wing(wing_id, *, area=None, aspect_ratio=None, taper=None):
    # drive the section by AR, area and taper; span and chords follow
    vsp.SetDriverGroup(wing_id, 1, vsp.AR_WSECT_DRIVER, vsp.AREA_WSECT_DRIVER, vsp.TAPER_WSECT_DRIVER)
    if area is not None:
        vsp.SetParmVal(wing_id, "Area", "XSec_1", area / 2)
    if aspect_ratio is not None:
        vsp.SetParmVal(wing_id, "Aspect", "XSec_1", aspect_ratio / 2)
    if taper is not None:
        vsp.SetParmVal(wing_id, "Taper", "XSec_1", taper)
    vsp.Update()


def parasite_drag(s_ref):
    # inputs saved in the file, reference area = current wing area
    an = "ParasiteDrag"
    vsp.SetAnalysisInputDefaults(an)
    settings = vsp.FindContainer("ParasiteDragSettings", 0)
    saved = {vsp.GetParmName(pid): pid for pid in vsp.FindContainerParmIDs(settings)}
    for name in vsp.GetAnalysisInputNames(an):
        if name not in saved:
            continue
        kind = vsp.GetAnalysisInputType(an, name)
        if kind == vsp.INT_DATA:
            vsp.SetIntAnalysisInput(an, name, [int(vsp.GetParmVal(saved[name]))])
        elif kind == vsp.DOUBLE_DATA:
            vsp.SetDoubleAnalysisInput(an, name, [vsp.GetParmVal(saved[name])])
    vsp.SetIntAnalysisInput(an, "GeomSet", [GEOM_SET])
    vsp.SetIntAnalysisInput(an, "RefFlag", [0])
    vsp.SetDoubleAnalysisInput(an, "Sref", [s_ref])

    t0 = time.perf_counter()
    rid = vsp.ExecAnalysis(an)
    print(f"Parasite drag took {time.perf_counter() - t0:.2f} s")

    labels = vsp.GetStringResults(rid, "Comp_Label", 0)
    components = pd.DataFrame({
        "S_wet [m²]": vsp.GetDoubleResults(rid, "Comp_Swet", 0),
        "f [m²]": vsp.GetDoubleResults(rid, "Comp_f", 0),
        "CD": vsp.GetDoubleResults(rid, "Comp_CD", 0),
    }, index=labels)
    return {
        "CD0": vsp.GetDoubleResults(rid, "Total_CD_Total", 0)[0],
        "f [m²]": vsp.GetDoubleResults(rid, "Total_f_Total", 0)[0],
        "components": components,
    }

## Baseline (file as saved)

In [3]:
wing_id = load_model()
before = wing_summary(wing_id)
drag_before = parasite_drag(before["area [m²]"])

display(before.round(4).to_frame("baseline"))
display(drag_before["components"].round(5))
print(f"CD0 = {drag_before['CD0']:.5f}  (S_ref = {before['area [m²]']:.3f} m²)")

Parasite drag took 0.97 s


,baseline
area [m²],0.4000
aspect ratio,8.0000
span [m],1.7889
mean chord [m],0.2236
MAC [m],0.2343
root chord [m],0.3084
tip chord [m],0.1388
taper,0.4500


,S_wet [m²],f [m²],CD
ARDEA_D3_Stack,0.77873,0.00272,0.00681
vertical stabilizer,0.06139,0.00039,0.00097
Horizontal stabilizer,0.11930,0.00075,0.00188
Main Wing,0.65477,0.00345,0.00862


CD0 = 0.01828  (S_ref = 0.400 m²)


## Modified wing

In [4]:
set_wing(wing_id, area=NEW_AREA, aspect_ratio=NEW_AR, taper=NEW_TAPER)
after = wing_summary(wing_id)
drag_after = parasite_drag(after["area [m²]"])

display(drag_after["components"].round(5))

comparison = pd.DataFrame({"baseline": before, "modified": after})
comparison.loc["CD0 (S_ref = wing area)"] = [drag_before["CD0"], drag_after["CD0"]]
comparison.loc["f = CD0·S [m²]"] = [drag_before["f [m²]"], drag_after["f [m²]"]]
comparison["change [%]"] = 100 * (comparison["modified"] / comparison["baseline"] - 1)
comparison.round(4)

Parasite drag took 1.57 s


,S_wet [m²],f [m²],CD
ARDEA_D3_Stack,0.77195,0.00270,0.00540
vertical stabilizer,0.06139,0.00039,0.00077
Horizontal stabilizer,0.11930,0.00075,0.00150
Main Wing,0.83667,0.00427,0.00853


,baseline,modified,change [%]
area [m²],0.4000,0.5000,25.0000
aspect ratio,8.0000,7.0000,-12.5000
span [m],1.7889,1.8708,4.5825
mean chord [m],0.2236,0.2673,19.5229
MAC [m],0.2343,0.2801,19.5229
root chord [m],0.3084,0.3686,19.5229
tip chord [m],0.1388,0.1659,19.5229
taper,0.4500,0.4500,0.0000
CD0 (S_ref = wing area),0.0183,0.0162,-11.3168
f = CD0·S [m²],0.0073,0.0081,10.8540


In [5]:
if SAVE_AS:
    vsp.WriteVSPFile(str(Path(SAVE_AS).resolve()))
    print(f"Saved to {SAVE_AS}")